# 04 - EfficientNet-B0 & Two-Phase Fine-Tuning

English: Transfer learning on pretrained EfficientNet-B0 with a true two-phase procedure:
1. **Phase 1**: Classifier head training with frozen backbone & BatchNorm in eval mode.
2. **Phase 2**: Deepest feature blocks (blocks 5-8) unfrozen with reduced learning rate (3e-5) & Cosine Annealing.

Hindi: Pretrained EfficientNet-B0 ke liye do-phase fine-tuning procedure.

## 1. Model Inspection & Parameter Groups

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from models import build_efficientnet, set_backbone_trainable, unfreeze_last_blocks

model = build_efficientnet(num_classes=2, pretrained=True)
print("EfficientNet-B0 Classifier Head:", model.classifier)

# Check Phase 1 vs Phase 2 trainable parameters
set_backbone_trainable(model, trainable=False)
head_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 Trainable Parameters (Head only): {head_params:,}")

unfrozen = unfreeze_last_blocks(model, from_block=5)
total_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 Trainable Parameters (Head + Blocks 5-8): {total_p2:,}")
print("Unfrozen blocks:", unfrozen)

## 2. Training Execution Commands

Run binary transfer learning:
```bash
python src/train_classifier.py --stage binary --model efficientnet --finetune
```

Run defect-type transfer learning:
```bash
python src/train_classifier.py --stage defect_type --model efficientnet --finetune
```